# 17 · Chromatic dispersion

In a fiber, the group velocity depends on the wavelength.  A signal of
bandwidth `B` occupies a range of wavelengths, its spectral components
arrive at different times, and each symbol smears over its neighbours.
Over a long link, **chromatic dispersion** (CD) spreads a symbol over
hundreds of symbol periods: by far the largest linear impairment of a
coherent link, and one that a coherent receiver removes almost exactly in
DSP.

1. the size of the effect: the dispersion parameter `D` and the spread;
2. the all-pass transfer function: dispersion changes the phase, not the
   power spectrum;
3. electronic **compensation** (CDC) and its cost;
4. **residual** dispersion, left to the adaptive equalizer.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import RRC, Constellation
from commkit.backend import to_device

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()
C16 = Constellation.qam(16)
RS = 32e9
D = 17.0  # ps/(nm km), standard single-mode fiber at 1550 nm
LAMBDA = 1550.0  # nm
C_LIGHT = 299_792_458.0


def host(a):
    return to_device(a, "cpu")


tx = ck.generate(
    C16,
    num_symbols=2**15,
    symbol_rate=RS,
    sps=2,
    pulse=RRC(rolloff=0.1),
    rng=1,
    device=DEVICE,
)


def cd(x, km):
    return ck.impairments.apply_chromatic_dispersion(
        x, dispersion_ps_nm_km=D, fiber_length_km=km, center_wavelength_nm=LAMBDA
    )


def cdc(x, km):
    return ck.filtering.correct_chromatic_dispersion(
        x, dispersion_ps_nm_km=D, fiber_length_km=km, center_wavelength_nm=LAMBDA
    )


def snr_after(x):
    return ck.metrics.snr(
        ck.multirate.decimate_to_symbol_rate(ck.filtering.matched_filter(x))
    )

## 1. How large is it?

The dispersion parameter `D` (17 ps/(nm·km) for standard fiber at
1550 nm) is the group-delay difference per nm of wavelength and per km.
A signal of bandwidth `B` spans `Δλ = λ² B / c`, so after a length `L` its
components are spread over

$$\Delta\tau = |D|\, L\, \Delta\lambda = |D|\, L\, \frac{\lambda^2 B}{c}.$$

In [ ]:
bw = (1 + 0.1) * RS
dlam_nm = (LAMBDA * 1e-9) ** 2 * bw / C_LIGHT * 1e9
print(f"bandwidth {bw / 1e9:.1f} GHz -> Δλ = {dlam_nm:.3f} nm")
for km in (1, 80, 1000):
    spread = D * km * dlam_nm * 1e-12
    print(
        f"L = {km:5d} km: spread {spread * 1e9:7.3f} ns = {spread * RS:7.1f} symbol periods"
    )

## 2. An all-pass filter

In the frequency domain, dispersion multiplies the spectrum by a phase
that is quadratic in frequency,

$$H(f) = \exp\!\left(-j\, \frac{\beta_2 L}{2} (2\pi f)^2\right), \qquad
\beta_2 = -\frac{D \lambda^2}{2\pi c},$$

with `|H(f)| = 1`: the power spectrum is unchanged and no energy is lost.
Everything happens in the phase, which is why CD is invisible on a power
spectrum and yet destroys the waveform.

In [ ]:
rx80 = cd(tx, 80)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ck.plotting.plot_psd(tx, nperseg=1024, ax=axes[0], title=None, label="transmitted")
ck.plotting.plot_psd(
    rx80,
    nperseg=1024,
    ax=axes[0],
    title="Power spectrum: unchanged",
    label="after 80 km",
)
axes[0].legend()
# One isolated pulse: transmit a single symbol and watch it spread.
pulse = tx.xp.zeros(4096, dtype=tx.samples.dtype)
pulse[2048] = 1
shaped = ck.filtering.fir_filter(pulse, tx.pulse.taps(2))
t = (np.arange(4096) - 2048) / 2  # symbol periods
for km in (0, 20, 80):
    y = (
        shaped
        if km == 0
        else ck.impairments.apply_chromatic_dispersion(
            shaped,
            dispersion_ps_nm_km=D,
            fiber_length_km=km,
            center_wavelength_nm=LAMBDA,
            sampling_rate=2 * RS,
        )
    )
    axes[1].plot(
        t, np.abs(host(y)) ** 2 / np.max(np.abs(host(shaped)) ** 2), label=f"{km} km"
    )
axes[1].set_xlim(-15, 15)
axes[1].set_xlabel("Time [symbol periods]")
axes[1].set_ylabel("|pulse|² (normalized)")
axes[1].set_title("One RRC pulse")
axes[1].legend()
plt.show()

for km in (1, 5, 20, 80):
    print(f"{km:3d} km uncompensated: SNR {snr_after(cd(tx, km)):6.2f} dB (noise-free)")

## 3. Compensation

Because `H(f)` is known and all-pass, its inverse is exact:
`correct_chromatic_dispersion` multiplies by `H*(f)`.  The function
applies it in the frequency domain (one FFT pair over the record), which is
how receivers do it, in overlap-save blocks.  The time-domain equivalent
would be an FIR filter as long as the spread, about `Δτ · fs` taps: the cost
grows linearly with the link length, which is why CDC is a dedicated block
in front of a short adaptive equalizer.

In [ ]:
for km in (80, 1000):
    rx = cd(tx, km)
    spread_samples = D * km * dlam_nm * 1e-12 * tx.sampling_rate
    print(
        f"{km:5d} km: before {snr_after(rx):7.2f} dB, after CDC {snr_after(cdc(rx, km)):6.2f} dB "
        f"(an FIR would need about {spread_samples:.0f} taps)"
    )

Two details:

* the sign convention: applying the compensation of the wrong sign doubles
  the dispersion instead of removing it, so the parameters are those of the
  fiber, for both the channel and the compensation;
* `center_wavelength_nm` matters because `β₂ ∝ D λ²`.

## 4. Residual dispersion

The link length (or the fiber's exact `D`) is rarely known to the last
kilometre.  A CDC for the nominal length leaves a small residual, which
the adaptive equalizer ([14](14-adaptive-equalization.ipynb)) absorbs if it has enough taps.
Here an 80 km link is compensated for 70 to 80 km, and a 15-tap LMS (7.5
symbols at 2 samples per symbol) cleans up the rest:

In [ ]:
rx = ck.impairments.apply_awgn(cd(tx, 80), esn0_db=20, rng=2)
print("CDC length   CDC only   CDC + 15-tap LMS")
for km in (80, 78, 75, 70):
    y = ck.filtering.matched_filter(cdc(rx, km))
    lms = ck.equalization.lms(
        y, num_taps=15, step_size=3e-3, training_symbols=y.reference.symbols[..., :4000]
    )
    print(
        f"{km:7d} km   {ck.metrics.snr(ck.multirate.decimate_to_symbol_rate(y)):7.2f} dB   "
        f"{ck.metrics.snr(lms.signal, num_skip_symbols=8000):9.2f} dB"
    )

**Summary**

* The spread is `|D| L λ² B / c`: about 12 symbol periods per 80 km at 32 GBd.
* CD is an all-pass filter: the power spectrum stays, the waveform does
  not.
* `correct_chromatic_dispersion` is its exact inverse, applied in the
  frequency domain; the adaptive equalizer handles the residual.
* The fiber's nonlinearity couples with dispersion at high power; that is
  the next notebook.

**Next:** [18 · Nonlinear fiber and digital backpropagation](18-nonlinear-fiber-and-backpropagation.ipynb).